# Assignment 5: Part I - Base Simulation (No Management)

This notebook uses the newly developed `fvs_tools` library to run FVS simulations on Section 6 plots (99-101, 293-297) for 100 years with no management interventions.

In [ ]:
# Import libraries
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path for importing fvs_tools
sys.path.insert(0, str(Path.cwd().parent / "src"))

# Enable autoreload for development (prevents stale module issues)
%load_ext autoreload
%autoreload 2

import fvs_tools as fvs

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style("whitegrid")

## Step 1: Load and Filter Data

Load the Lubrecht 2023 inventory data and filter to Section 6 plots (99-101, 293-297).

In [ ]:
# Load full dataset
stands_full = fvs.load_stands()
trees_full = fvs.load_trees()

print(f"Full dataset: {len(stands_full)} stands, {len(trees_full)} trees")

# Filter to Section 6 plots
section6_plots = [99, 100, 101, 293, 294, 295, 296, 297]
stands_sec6, trees_sec6 = fvs.filter_by_plot_ids(stands_full, trees_full, section6_plots)

print(f"\nSection 6: {len(stands_sec6)} stands, {len(trees_sec6)} trees")
print(f"Stand IDs: {', '.join(stands_sec6['STAND_ID'].tolist())}")

# Display stand summary
stands_sec6[["STAND_ID", "PlotID", "INV_YEAR", "ASPECT", "SLOPE", "ELEVFT", "Strata"]].head(10)

## Step 2: Configure Base Simulation

Set up a 100-year projection (2023-2123) with:
- Calibration enabled
- No tripling
- No regeneration
- Carbon and fuels output
- Canopy cover computation

In [ ]:
# Create simulation configuration
config = fvs.FVSSimulationConfig(
    name="base_no_management",
    num_years=100,
    cycle_length=10,
    output_treelist=True,
    output_carbon=True,
    compute_canopy_cover=True,
)

print(f"Simulation: {config.name}")
print(f"Run ID: {config.run_id}")
print(f"Duration: {config.num_years} years ({config.num_cycles} cycles)")
print(f"Canopy cover tracking: {config.compute_canopy_cover}")

## Step 3: Run Batch Simulation

Execute FVS for all 8 Section 6 stands.

In [ ]:
# Set output directory
output_dir = Path(f"../outputs/assignment5_part1_base_{config.run_id}")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Create FVS input database (scales better for large runs)
input_db = output_dir / "FVS_Data.db"
fvs.create_fvs_input_db(stands_sec6, trees_sec6, input_db)

# Run batch simulation using database input
results = fvs.run_batch_simulation(
    stands=stands_sec6,
    trees=trees_sec6,
    config=config,
    output_base=output_dir,
    use_database=True,
    input_database=input_db
)

# Check run status
print("\nRun Status:")
status_df = results["run_status"]
print(f"Total runs: {len(status_df)}")
print(f"Successful: {status_df['success'].sum()}")
print(f"Failed: {len(status_df) - status_df['success'].sum()}")

if not status_df["success"].all():
    print("\nFailed runs:")
    print(status_df[~status_df["success"]])
else:
    print("All runs completed successfully.")

## Part I:1 - Calibration Statistics

Examine the calibration applied by FVS to diameter growth, height growth, and mortality.

In [ ]:
# Display calibration statistics by stand
if "calibration_stats" in results and results["calibration_stats"] is not None:
    calib = results["calibration_stats"]
    
    # Select key columns for display
    display_cols = ["StandID", "SpeciesFVS", "TreeSize", "NumTrees", "ScaleFactor", "ReadCorMult"]
    available_cols = [c for c in display_cols if c in calib.columns]
    
    # Remove duplicate rows based on the display columns
    calib_unique = calib[available_cols].drop_duplicates()
    
    print("="*80)
    print("PART I, QUESTION 1: CALIBRATION ANALYSIS")
    print("="*80)
    
    print("\n** Type of Calibration **")
    print("FVS applied AUTOMATIC SELF-CALIBRATION based on measured growth data.")
    print("This calibration adjusts FVS's regional growth equations to match observed")
    print("tree growth patterns from the Lubrecht inventory data.")
    
    print("\n** Calibration Statistics by Stand and Species **")
    display(calib_unique.round(2))
    
    # Categorize stands
    slow_growth = calib_unique[calib_unique["ScaleFactor"] < 0.8]
    fast_growth = calib_unique[calib_unique["ScaleFactor"] > 1.2]
    normal_growth = calib_unique[(calib_unique["ScaleFactor"] >= 0.8) & (calib_unique["ScaleFactor"] <= 1.2)]
    
    print("FVS applied data-driven diameter growth calibration to adjust its Inland")
    print("Empire variant predictions to match observed growth rates. Calibration")
    print("multipliers (ScaleFactor) ranged widely, indicating that actual tree growth")
    print("in Section 6 varies considerably from FVS regional defaults.")
else:
    print("No calibration statistics found in output.")

In [ ]:
# Check for FVS errors across the batch
print("=== FVS Error Messages (All Stands) ===")
errors = fvs.collect_batch_errors(output_dir)
if errors.empty:
    print("  No errors found - all runs completed successfully!")
else:
    print(f"  Found {len(errors)} error message(s):")
    for _, row in errors.iterrows():
        print(f"  [{row['StandID']}] {row['Message']}")

## Part I:2 - Summary by 10-Year Period

Calculate average basal area, aboveground carbon, standing dead carbon, and canopy cover across all 8 plots for each 10-year period.

In [ ]:
# Aggregate results by period
summary_by_period = fvs.aggregate_by_period(
    summary_df=results["summary_all"],
    carbon_df=results.get("carbon_all"),
    compute_df=results.get("compute_all"),
    harvest_carbon_df=results.get("harvest_carbon_all"),
    years_per_period=10
)

# Display available data
print("PART I, QUESTION 2: Average Metrics by 10-Year Period (across 8 plots)")
display(summary_by_period.round(2))

final_ba = summary_by_period[summary_by_period['Year'] == 2123]['BA'].values[0]
print(f"\nFinal year (2123) average BA: {final_ba:.2f} ft²/ac")
print("Expected: ~161 ft²/ac")

## Visualizations

Plot trends over time for key metrics.

In [ ]:
# Create visualization of key metrics over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Basal Area
ax = axes[0, 0]
ax.plot(summary_by_period["Year"], summary_by_period["BA"], marker='o', linewidth=2)
ax.set_xlabel("Year")
ax.set_ylabel("Basal Area (ft²/ac)")
ax.set_title("Average Basal Area Over Time")
ax.grid(True, alpha=0.3)

# Aboveground Carbon
if "Aboveground_C_Live" in summary_by_period.columns:
    ax = axes[0, 1]
    ax.plot(summary_by_period["Year"], summary_by_period["Aboveground_C_Live"], 
            marker='o', linewidth=2, color='green')
    ax.set_xlabel("Year")
    ax.set_ylabel("Aboveground C (tons/ac)")
    ax.set_title("Average Live Tree Carbon")
    ax.grid(True, alpha=0.3)

# Standing Dead Carbon
if "Standing_Dead_C" in summary_by_period.columns:
    ax = axes[1, 0]
    ax.plot(summary_by_period["Year"], summary_by_period["Standing_Dead_C"], 
            marker='o', linewidth=2, color='brown')
    ax.set_xlabel("Year")
    ax.set_ylabel("Standing Dead C (tons/ac)")
    ax.set_title("Average Standing Dead Carbon")
    ax.grid(True, alpha=0.3)

# Canopy Cover
if "Canopy_Cover_Pct" in summary_by_period.columns:
    ax = axes[1, 1]
    ax.plot(summary_by_period["Year"], summary_by_period["Canopy_Cover_Pct"], 
            marker='o', linewidth=2, color='darkgreen')
    ax.axhline(y=40, color='red', linestyle='--', label='40% threshold')
    ax.set_xlabel("Year")
    ax.set_ylabel("Canopy Cover (%)")
    ax.set_title("Average Canopy Cover")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Export Results

Save the summary table for comparison with GUI results.

In [ ]:
# Save summary table to CSV
output_csv = output_dir / "part1_summary_by_period.csv"
summary_by_period.to_csv(output_csv, index=False)
print(f"Summary table saved to: {output_csv}")

# Also save full aggregated results
results["summary_all"].to_csv(output_dir / "summary_all_stands.csv", index=False)
if "carbon_all" in results:
    results["carbon_all"].to_csv(output_dir / "carbon_all_stands.csv", index=False)
if "compute_all" in results:
    results["compute_all"].to_csv(output_dir / "compute_all_stands.csv", index=False)

print("\nAll results exported to:", output_dir)

## Part II: Uneven-aged Management Simulation

Now we implement the "harv1" scenario:
- **Thinning:** Q-factor 2.0, Residual BA 65, Trigger BA 100, DBH 2-24"
- **Harvest:** Minimum harvest volume 4500 bdft/ac (applied at start)


In [ ]:
# Configure Harvest Scenario (harv1)
import shutil

config_harv = fvs.FVSSimulationConfig(
    name="harv1",
    num_years=100,
    cycle_length=10,
    output_treelist=True,
    output_carbon=True,
    compute_canopy_cover=True,
    # Management Parameters
    thin_q_factor=2.0,
    thin_residual_ba=65.0,
    thin_trigger_ba=100.0,
    thin_min_dbh=2.0,
    thin_max_dbh=24.0,
    min_harvest_volume=4500.0
)

print(f"Scenario: {config_harv.name}")
print(f"Run ID: {config_harv.run_id}")
print(f"Thinning: Q={config_harv.thin_q_factor}, ResBA={config_harv.thin_residual_ba}, Trigger={config_harv.thin_trigger_ba}")
print(f"Min Harvest: {config_harv.min_harvest_volume} bdft/ac")

# Run Batch Simulation for Harvest Scenario
output_dir_harv = Path(f"../outputs/assignment5_part2_harv1_{config_harv.run_id}")

# Clear previous output (if any, though run_id makes it unique)
if output_dir_harv.exists():
    print(f"Clearing previous output: {output_dir_harv}")
    shutil.rmtree(output_dir_harv)
output_dir_harv.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir_harv}")

# Create FVS input database for harvest scenario
input_db_harv = output_dir_harv / "FVS_Data.db"
fvs.create_fvs_input_db(stands_sec6, trees_sec6, input_db_harv)
print(f"Created input database: {input_db_harv}")

# Run using database input (scalable for 1000+ runs)
results_harv = fvs.run_batch_simulation(
    stands=stands_sec6,
    trees=trees_sec6,
    config=config_harv,
    output_base=output_dir_harv,
    use_database=True,
    input_database=input_db_harv
)

print("\nRun Status:")
status_df = results_harv["run_status"]
print(f"Total runs: {len(status_df)}")
print(f"Successful: {status_df['success'].sum()}")
print(f"Failed: {len(status_df) - status_df['success'].sum()}")

if not status_df["success"].all():
    print("\nFailed runs:")
    print(status_df[~status_df["success"]])
else:
    print("All runs completed successfully.")

## Part II:3 - Harvest Scenario Summary

Tabulate metrics including Harvested Carbon and Cumulative Removals.


In [ ]:
# Aggregate results for Harvest Scenario (including harvest carbon)
summary_harv = fvs.aggregate_by_period(
    summary_df=results_harv["summary_all"],
    carbon_df=results_harv.get("carbon_all"),
    compute_df=results_harv.get("compute_all"),
    harvest_carbon_df=results_harv.get("harvest_carbon_all"),
    years_per_period=10
)

# Calculate cumulative timber removals AFTER averaging (so it's monotonically increasing)
if "RBdFt" in summary_harv.columns:
    summary_harv = summary_harv.sort_values("Year")
    summary_harv["Cumulative_RBdFt"] = summary_harv["RBdFt"].cumsum()

# NOTE: Merch_Carbon_Stored is already a POOL BALANCE (cumulative with decay)
# It represents carbon currently stored in wood products at each time step,
# NOT a per-period addition. Do NOT apply cumsum() to it!

# Build display columns list based on what's available
cols_to_show = ["Year"]
for col in ["BA", "Aboveground_C_Live", "Standing_Dead_C", "Canopy_Cover_Pct", 
            "Merch_Carbon_Stored", "RBdFt", "Cumulative_RBdFt"]:
    if col in summary_harv.columns:
        cols_to_show.append(col)

print("PART II, QUESTION 3: Harvest Scenario Metrics (Average across 8 plots)")
print("Merch_Carbon_Stored = Carbon currently stored in wood products pool (tons/ac)")
display(summary_harv[cols_to_show].round(2))

## Part II:4 - Evaluation and Comparison

Compare the Base and Harvest scenarios to answer:
- How does harvesting shift C stocks?
- Do we lose or gain C?
- Do we maintain 40% canopy cover?
- How much timber is extracted?


In [ ]:
# Comparison Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Total Carbon (Live + Dead + Harvested)
base_total_c = summary_by_period["Aboveground_C_Live"] + summary_by_period["Standing_Dead_C"]
harv_live_dead_c = summary_harv["Aboveground_C_Live"] + summary_harv["Standing_Dead_C"]

# Add stored carbon for harvest scenario
# Merch_Carbon_Stored is already the pool balance (carbon currently in wood products)
if "Merch_Carbon_Stored" in summary_harv.columns:
    harv_total_c = harv_live_dead_c + summary_harv["Merch_Carbon_Stored"]
else:
    harv_total_c = harv_live_dead_c

ax = axes[0, 0]
ax.plot(summary_by_period["Year"], base_total_c, label="Base (Live+Dead)", marker='o')
ax.plot(summary_harv["Year"], harv_total_c, label="Harv (Live+Dead+Stored)", marker='s')
ax.set_title("Total Carbon Stocks")
ax.set_ylabel("Carbon (tons/ac)")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Live Tree Carbon
ax = axes[0, 1]
ax.plot(summary_by_period["Year"], summary_by_period["Aboveground_C_Live"], label="Base", marker='o', color='green')
ax.plot(summary_harv["Year"], summary_harv["Aboveground_C_Live"], label="Harv", marker='s', color='lightgreen')
ax.set_title("Live Tree Carbon")
ax.set_ylabel("Carbon (tons/ac)")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Canopy Cover
ax = axes[1, 0]
ax.plot(summary_by_period["Year"], summary_by_period["Canopy_Cover_Pct"], label="Base", marker='o', color='darkgreen')
ax.plot(summary_harv["Year"], summary_harv["Canopy_Cover_Pct"], label="Harv", marker='s', color='orange')
ax.axhline(y=40, color='red', linestyle='--', label='40% Threshold')
ax.set_title("Canopy Cover")
ax.set_ylabel("Percent (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Cumulative Timber Removal
ax = axes[1, 1]
if "Cumulative_RBdFt" in summary_harv.columns:
    ax.plot(summary_harv["Year"], summary_harv["Cumulative_RBdFt"], label="Harv", marker='s', color='brown')
ax.set_title("Cumulative Timber Harvest")
ax.set_ylabel("Board Feet / ac")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Merch_Carbon_Stored is the carbon currently in the wood products pool (with decay)
stored_merch_c = summary_harv["Merch_Carbon_Stored"].iloc[-1] if "Merch_Carbon_Stored" in summary_harv.columns else 0
cumulative_bdft = summary_harv["Cumulative_RBdFt"].iloc[-1] if "Cumulative_RBdFt" in summary_harv.columns else 0

comparison = pd.DataFrame({
    "Metric": ["Live + Dead C (tons/ac)", "Live C (tons/ac)", "Dead C (tons/ac)", 
               "Stored Merch C (tons/ac)", "Total C incl. Stored", 
               "Canopy Cover (%)", "Cumulative Harvest (bdft/ac)"],
    "Base (2123)": [
        base_total_c.iloc[-1],
        summary_by_period["Aboveground_C_Live"].iloc[-1],
        summary_by_period["Standing_Dead_C"].iloc[-1],
        0,
        base_total_c.iloc[-1],
        summary_by_period["Canopy_Cover_Pct"].iloc[-1],
        0
    ],
    "Harvest (2123)": [
        harv_live_dead_c.iloc[-1],
        summary_harv["Aboveground_C_Live"].iloc[-1],
        summary_harv["Standing_Dead_C"].iloc[-1],
        stored_merch_c,
        harv_total_c.iloc[-1],
        summary_harv["Canopy_Cover_Pct"].iloc[-1],
        cumulative_bdft
    ]
})
comparison["Difference"] = comparison["Harvest (2123)"] - comparison["Base (2123)"]

print("\nPART II, QUESTION 4: Final Year Comparison (2123)")
display(comparison.round(2))

### Discussion

**1. Shift in Carbon Stocks:**
The harvesting regime significantly reduces live tree carbon stocks compared to the base scenario (~20 vs ~48 tons/ac by 2123). While some carbon is transferred to long-term storage in wood products (~15 tons/ac), the total carbon (Live + Dead + Stored ≈ 38 tons/ac) remains lower than the unmanaged base scenario (~51 tons/ac). This seems reasonable to me, with actively managed/harvested forest you can get a harvest and after 100 years have a similar total carbon to when you started, but much less than if you had not harvested at all.

**2. Canopy Cover:**
Our thinning puts us below the 40% canopy cover at two points. I imagine with some small tweaks we could get a similar end outcome without dipping below though.

**3. Timber Extraction:**
We end up harvesting 17000 bdft/ac under these assumptions.


## Export to HTML

Export the completed notebook to HTML for submission.

In [ ]:
import subprocess
from datetime import datetime

# Get the notebook path
notebook_path = Path.cwd() / "Assignment5.ipynb"

# Create output filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_output = output_dir.parent / f"Assignment5_{timestamp}.html"

# Export to HTML using nbconvert
result = subprocess.run(
    ["uv", "run", "jupyter", "nbconvert", "--to", "html", 
     "--output", str(html_output), str(notebook_path)],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"✓ Notebook exported successfully!")
    print(f"  Output: {html_output}")
else:
    print(f"✗ Export failed:")
    print(result.stderr)